In [1]:
import numpy as np
import cv2

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.widgets import Button, Slider
import matplotlib.gridspec as gridspec
import simpleaudio as sa

import sys
sys.path.append("..//neural/")
import format_waveform_data, waveform_analysis
sys.path.append("..//utils/")
from helpers import nan_interp
from load_matlab_data import loadmat_sbx
from video_utils import PySBA, crop_bird 

import scipy.io
import os
import mat73

In [2]:
''' set paths '''
root_dir = "Z:/Isabel/data/"
bird_id = 'SLV132'
session_id = '250310'
locker_root = f"{root_dir}hpc_implants/{bird_id}/{bird_id}_{session_id}/"

# to load waveforms and probe info
# local_root = "../data/antidromic_stim/" # local
ephys_id = "SLV132_250310_101340"
ks_dir = "kilosort4/"
session_dir = f"{locker_root}/{ephys_id}/" # locker
# session_dir = f"{local_root}{bird_id}/" # local
ephys_dir = f"{session_dir}raw_ephys_output/"

# specify broken channels for this session
'''
AMB154_241119 - broken_channels = np.asarray([3, 23, 28, 29, 41, 48, 50, 61, 63])
RBY94_241129 - broken_channels = np.asarray([14, 22, 35, 42, 48])
SLV132_250303 - broken_channels = np.asarray([53, 56, 57, 63])
'''
broken_channels = np.asarray([53, 56, 63])

# to load raw keypoints, model performance measures, behavior
pred_file = f'250310_posture_2stage_face.npy'
data_dir = f"{locker_root}behavior_data/"
pred_path = f"{data_dir}{pred_file}"
annotated_seed_file = "annotatedSeeds.mat"

# to load video files and camera params
cam_ids = ['red_cam', 'yellow_cam', 'green_cam', 'blue_cam']
n_cams = len(cam_ids)
video_paths = []
for i in range(len(cam_ids)):
    cam = cam_ids[i]
    video_paths.append(f"{locker_root}{cam}.avi")

In [3]:
# set save folder
save_plots = False
save_dir = f"../figures/basic_neural_analysis/{bird_id}/"
if os.path.isdir(save_dir):
    print('save directory exists')
else:
    os.mkdir(save_dir)
save_folder = f"{save_dir}/{bird_id}_{session_id}/"
if os.path.isdir(save_folder):
    print('save folder exists')
else:
    os.mkdir(save_folder)
save_subfolder = f"{save_folder}video/"
if os.path.isdir(save_subfolder):
    print('save sub-folder exists')
else:
    os.mkdir(save_subfolder)

save directory exists
save folder exists
save sub-folder exists


In [4]:
''' probe info '''
sampling_rate = 30000
n_channels_shank = 32
A_shank = np.arange(n_channels_shank)
B_shank = np.arange(n_channels_shank) + n_channels_shank
A_shank = np.setdiff1d(A_shank, broken_channels)
B_shank = np.setdiff1d(B_shank, broken_channels)

In [5]:
''' load and format the waveform struct '''
waveform_struct = format_waveform_data.load_wf_data(session_dir, ks_dir=ks_dir)
wf_ids = waveform_struct['goodIDs']
mean_waveforms, wf_channels, _, ch_names = format_waveform_data.sort_wf_by_channel('', waveform_struct,
                                                                                   data_dir=ephys_dir,
                                                                                   return_ch_names=True)

Z:/Isabel/data/hpc_implants/SLV132/SLV132_250310//SLV132_250310_101340/kilosort4/waveformStruct.mat
dict_keys(['mxWF', 'max_site', 'pcWF', 'meanRate', 'waveFormsMean', 'spkDur', 'spkOffset', 'goodIDs', 'goodLabels', 'medISI', 'contam', 'nSpikes'])
Z:/Isabel/data/hpc_implants/SLV132/SLV132_250310//SLV132_250310_101340/raw_ephys_output/intan_info.mat


In [6]:
# filter out the bad channels and get the new channel index
all_ch = np.arange(mean_waveforms.shape[1])
good_idx = np.setdiff1d(all_ch, broken_channels)
mean_waveforms = mean_waveforms[:, good_idx]
ch_names = [ch_names[i] for i in good_idx]
wf_ch_idx = np.asarray([ch_names.index(ch) for ch in wf_channels])
n_cells = mean_waveforms.shape[0]

# get the cell IDs and raw spike times
good_clusters, spike_id, spike_samp_raw = format_waveform_data.get_spike_times(session_dir, ks_dir=ks_dir)
assert good_clusters.shape[0] == n_cells

In [7]:
# get the session average firing rate
avg_firing_rate = waveform_struct['meanRate']

In [8]:
''' load and format the annotated seed struct '''
seed_struct = loadmat_sbx(f'{data_dir}{annotated_seed_file}')['annotatedSeeds']
count_data = seed_struct['countData']
print(seed_struct.keys())
print(count_data.keys())

Z:/Isabel/data/hpc_implants/SLV132/SLV132_250310/behavior_data/annotatedSeeds.mat
dict_keys(['countData', 'bk_height_seedDetect', 'smSeedWindow', 'gainThresh', 'loseThresh', 'minLoseDur', 'validFrames', 'seedIntTol', 'path', 'beakPos', 'smSeed', 'cacheLoc', 'seedTimes', 'nEvents', 'newCacheTimes', 'endCacheTimes', 'cacheNum', 'fromCache_site', 'fromCache_event', 'toCache_site', 'toCache_event', 'gainMat', 'loseMat', 'initImgs', 'preIms', 'postIms', 'flagInt', 'seedChanges', 'initSeedCounts', 'prePreds', 'postPreds'])
dict_keys(['newPerch', 'endPerch', 'perchNum', 'newSite', 'endSite', 'siteNum', 'newFeeder', 'endFeeder', 'feederNum', 'newWater', 'endWater', 'waterNum', 'newBeakPerch', 'endBeakPerch', 'beakPerchNum', 'params'])


In [9]:
''' classify caches/retrievals/checks '''
# data params
all_int_start = count_data['newSite']
all_int_end = count_data['endSite']
all_int_changes = np.sum(seed_struct['seedChanges'], axis=1)
n_interactions = all_int_start.shape[0]

# caches = add a seed
cache_onsets = all_int_start[all_int_changes > 0]
cache_offsets = all_int_end[all_int_changes > 0]
caches = all_int_changes.copy()
caches[all_int_changes < 0] = 0

# retrievals = subtract a seed
retrieval_onsets = all_int_start[all_int_changes < 0]
retrieval_offsets = all_int_end[all_int_changes < 0]
retrievals = all_int_changes.copy()
retrievals[all_int_changes > 0] = 0

# checks = no change
check_onsets = all_int_start[all_int_changes == 0]
check_offsets = all_int_end[all_int_changes == 0]
checks = all_int_changes==0

In [10]:
''' get occupied vs. empty checks '''
n_checks = check_onsets.shape[0]

# for checks, intial seed count is known
seeds_in_sites = np.cumsum(seed_struct['seedChanges'], axis=0) + seed_struct['initSeedCounts']
interaction_site = count_data['siteNum'] - 1

# classify each check as occupied vs. empty
occupied_check = np.zeros(n_checks, dtype=bool)
check_idx = -1
for n_int in range(n_interactions):
    n_site = interaction_site[n_int]
    if checks[n_int]:
        check_idx += 1
    if seeds_in_sites[n_int, n_site] & checks[n_int]:
        occupied_check[check_idx] = True

In [12]:
''' load the frame times '''
framet_raw = np.load(f'{data_dir}frame_times.npy')
framet_raw = np.squeeze(framet_raw)
dt = np.unique(np.round(np.diff(framet_raw), 4))
if dt.shape[0] > 1:
    print(f'warning irregular frame times!\nframe dt = {dt}')
else:
    dt = dt[0]
    
assert seed_struct['validFrames'].shape[0] == framet_raw.shape[0]

AssertionError: 

In [13]:
framet_raw.shape[0]

385481

In [14]:
seed_struct['validFrames'].shape[0] - framet_raw.shape[0]

2546

In [15]:
# align so that 0 is the video start time
start_t = framet_raw[0]
frame_t = framet_raw - start_t
spike_t = spike_samp_raw - start_t*sampling_rate
frame_samples = np.append(frame_t, frame_t[-1] + dt)*sampling_rate

# keep only spikes from within the session
spike_id = spike_id[(spike_t >= 0) & (spike_t <= frame_samples[-1])]
spike_t = spike_t[(spike_t >= 0) & (spike_t <= frame_samples[-1])]

n_frames = frame_t.shape[0]

In [16]:
''' spikes per frame and spike bool '''
spike_fr = np.zeros((n_cells, n_frames))
i = -1
for c_idx, cell in enumerate(good_clusters):
    i += 1
    spk_times = spike_t[spike_id==cell]       
    spike_fr[i], _ = np.histogram(spk_times, frame_samples)
spike_bool = spike_fr.astype(bool)

In [17]:
''' audio for spikes '''
def generate_pop():
    duration_ms = 20
    fs = 44100
    samples = int(fs * duration_ms / 1000)
    
    # Create a short sharp click/pop
    pop = np.zeros(samples, dtype=np.int16)
    pop[:5] = 30000  # sharp edge at beginning
    
    return sa.WaveObject(pop, 1, 2, fs)

pop_sound = generate_pop()

def play_pop():
    pop_sound.play()

In [18]:
''' load the predicted key points and camera params '''
raw_preds = np.load(pred_path, allow_pickle=True).item()

In [19]:
''' params for cropping '''
crop_size = (320, 320) # pixels
cam_params = raw_preds['cam_params']
com_pts = raw_preds['results']['com_preds']
sba = PySBA(cam_params, np.NaN, np.NaN, np.NaN, np.NaN)

In [20]:
%matplotlib qt

In [21]:
''' set data params '''
# define the cell to examine
cell_id = 12
c_idx = np.where(good_clusters==cell_id)[0][0]
spikes = spike_bool[c_idx]

# sort caches by duration
cache_dur = cache_offsets - cache_onsets
duration_idx = np.argsort(cache_dur)
dur_idx = 19

# define the cache event
cache_idx = duration_idx[dur_idx]

# time params
window_sec = 10 # size of the window around the current spike
total_time_sec = 30 # total video length, starting window_sec before the cache
window_size = int(window_sec // dt)
total_frames = int(total_time_sec // dt)
cache_on = cache_onsets[cache_idx]
start_frame = cache_on - window_size

In [22]:
''' video set-up '''
# set up videos
all_readers = []
for i in range(n_cams):
    cam = cam_ids[i]
    print(cam)
    camPath = f"{locker_root}{cam}.avi"

    # define the video reader obj and settings
    api_id = cv2.CAP_FFMPEG
    reader = cv2.VideoCapture(camPath, api_id)
    all_readers.append(reader)

# check number of frames
num_frames = int(all_readers[0].get(cv2.CAP_PROP_FRAME_COUNT))
# if len(spikes) != num_frames:
#     raise ValueError("Spike array length must match number of frames.")

# cue up to start frame    
for reader in all_readers:
    reader.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

red_cam
yellow_cam
green_cam
blue_cam


In [23]:
''' crop around bird and save frames '''
full_img = np.zeros((n_cams, 640, 1896)) # update as-needed to match image size
crop_imgs = np.zeros((n_cams, crop_size[0], crop_size[1], total_frames), dtype='uint8')
stopReading = False # flag for reading failure
for n_frame in range(total_frames):
    frame_idx = n_frame + start_frame
    
    # read in the frame for each camera
    for n_cam in range(n_cams):
        flag, img = all_readers[n_cam].read()
        if img is None:
            stopReading = True
            break    
        full_img[n_cam, :, :] = img[:, :, 0]
    if np.mod(n_frame, 100)==0:
        print(f'Reading Frame {n_frame}')
        
    # if reading for any video failed, terminate tracking
    if stopReading:
        print('Terminated Reading on Frame {}'.format(n_frame))
        break
        
    # crop around the bird using the COM body prediction
    body_COM = com_pts[frame_idx, 1]
    crop_imgs[:, :, :, n_frame], _, _ = crop_bird(full_img, body_COM, cam_params, sba)

Reading Frame 0
Reading Frame 100
Reading Frame 200
Reading Frame 300
Reading Frame 400
Reading Frame 500
Reading Frame 600
Reading Frame 700
Reading Frame 800
Reading Frame 900
Reading Frame 1000
Reading Frame 1100
Reading Frame 1200
Reading Frame 1300
Reading Frame 1400


In [24]:
''' plot video + spikes '''
fig = plt.figure(figsize=(10, 10))
gs = gridspec.GridSpec(4, 2, height_ratios=[1, 5, 5, 0.2])

# Spike plot elements
ax_spikes = fig.add_subplot(gs[0, :])
buffer = window_size//2
these_spikes = np.where(spikes[start_frame - buffer:start_frame + total_frames + buffer])[0] - buffer
spike_lines = ax_spikes.vlines(these_spikes, 0, 1, color='black', clip_on=True)
center_line = ax_spikes.axvline(0, color='red')
ax_spikes.set_ylim(0, 1)
ax_spikes.set_yticks([])

# container for videos
video_axes = [fig.add_subplot(gs[i+1, j]) for i in range(2) for j in range(2)]
ims = [ax.imshow(np.zeros(crop_size), cmap='gray', vmin=0, vmax=255) for ax in video_axes]
for ax in video_axes:
    ax.axis('off')
    
# video controls
ax_btn = fig.add_subplot(gs[3, 0])
ax_slider = fig.add_subplot(gs[3, 1])
btn = Button(ax_btn, 'Pause')
slider = Slider(ax_slider, 'Frame',
                start_frame, start_frame + total_frames,
                valinit=start_frame, valfmt='%d')

# initial state
is_paused = False
current_frame_idx = [0]

def update(_):
    if is_paused:
        return
    
    frame_pos = start_frame + current_frame_idx[0]
    if frame_pos >= num_frames:
        ani.event_source.stop()
        return
    
    # video
    for i in range(4):
        view = crop_imgs[i, :, :, current_frame_idx[0]]
        ims[i].set_data(view)
    
    # spike plot
    window_start = current_frame_idx[0] - window_size//2
    window_end = current_frame_idx[0] + window_size//2
    center_line.set_xdata(current_frame_idx[0])
    ax_spikes.set_xlim(window_start, window_end)
    ax_spikes.set_title(f"Frame: {frame_pos}")

    if spikes[frame_pos]:
        play_pop()
    slider.set_val(frame_pos)
    current_frame_idx[0] += 1

    return ims + [spike_lines, center_line]

# play/pause button
def toggle_pause(event):
    global is_paused
    is_paused = not is_paused
    btn.label.set_text('Play' if is_paused else 'Pause')

btn.on_clicked(toggle_pause)

# frame slider
def on_slider(val):
    current_frame_idx[0] = int(val) - start_frame

slider.on_changed(on_slider)

# plot animated figure
ani = FuncAnimation(fig, update, 
                    frames=np.arange(100),
                    cache_frame_data=False,
                    interval=1000*dt,
                    blit=False)

plt.tight_layout()
plt.show()

C:\Users\ilow1\AppData\Local\Temp\ipykernel_25808\2542148680.py:49: MatplotlibDeprecationWarning: Setting data with a non sequence type is deprecated since 3.7 and will be remove two minor releases later
  center_line.set_xdata(current_frame_idx[0])


In [29]:
plt.plot(np.cumsum(these_spikes))